# Student Academic Record Management System
**Mini-project: MySQL + Python + Pandas + Matplotlib**

This notebook connects to MySQL, creates the tables, inserts fictional demo records, demonstrates CRUD, runs SQL reports and draws three graphs.

## Setup
1. Create a MySQL server/database.
2. Run `student_academic_record.sql` in MySQL Workbench/provider console (or let the notebook create the tables below).
3. In Colab, open Secrets (key icon) and add `MYSQL_HOST`, `MYSQL_PORT`, `MYSQL_USER`, `MYSQL_PASSWORD`, `MYSQL_DATABASE`.
4. Run cells from top to bottom. For a remote server, allow network access only as needed and use TLS. Do not publish credentials.

## 1. Install packages

In [ ]:
%pip -q install mysql-connector-python pandas matplotlib

## 2. Load Colab Secrets

In [ ]:
from google.colab import userdata
import mysql.connector
import pandas as pd
import matplotlib.pyplot as plt

MYSQL_HOST = userdata.get("MYSQL_HOST")
MYSQL_PORT = int(userdata.get("MYSQL_PORT") or "3306")
MYSQL_USER = userdata.get("MYSQL_USER")
MYSQL_PASSWORD = userdata.get("MYSQL_PASSWORD")
MYSQL_DATABASE = userdata.get("MYSQL_DATABASE")

# Optional: if your provider requires a CA certificate, upload the .pem file
# in the next cell and assign its filename to MYSQL_SSL_CA.
MYSQL_SSL_CA = None
print("Configuration loaded; password is not displayed.")

## 3. Optional TLS certificate upload

In [ ]:
# Uncomment and run only if your provider gave you a CA certificate:
# from google.colab import files
# uploaded = files.upload()
# MYSQL_SSL_CA = next(iter(uploaded))

## 4. Connection helper and test

In [ ]:
def get_connection():
    config = {
        "host": MYSQL_HOST, "port": MYSQL_PORT, "user": MYSQL_USER,
        "password": MYSQL_PASSWORD, "database": MYSQL_DATABASE,
        "connection_timeout": 15
    }
    if MYSQL_SSL_CA:
        config.update({"ssl_ca": MYSQL_SSL_CA, "ssl_verify_cert": True,
                       "ssl_verify_identity": True})
    return mysql.connector.connect(**config)

conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT VERSION(), DATABASE()")
version, db_name = cur.fetchone()
print("Connected to MySQL", version, "| database:", db_name)
cur.close()
conn.close()

## 5. Create six tables (safe if already created)

In [ ]:
schema = [
"""CREATE TABLE IF NOT EXISTS courses (
course_id INT AUTO_INCREMENT PRIMARY KEY, course_code VARCHAR(20) NOT NULL UNIQUE,
course_name VARCHAR(100) NOT NULL) ENGINE=InnoDB""",
"""CREATE TABLE IF NOT EXISTS students (
student_id INT AUTO_INCREMENT PRIMARY KEY, roll_number VARCHAR(30) NOT NULL UNIQUE,
student_name VARCHAR(100) NOT NULL, email VARCHAR(150) UNIQUE, course_id INT NOT NULL,
created_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
UNIQUE KEY uq_student_course(student_id,course_id),
FOREIGN KEY(course_id) REFERENCES courses(course_id) ON UPDATE CASCADE ON DELETE RESTRICT
) ENGINE=InnoDB""",
"""CREATE TABLE IF NOT EXISTS subjects (
subject_id INT AUTO_INCREMENT PRIMARY KEY, subject_code VARCHAR(20) NOT NULL UNIQUE,
subject_name VARCHAR(100) NOT NULL) ENGINE=InnoDB""",
"""CREATE TABLE IF NOT EXISTS course_subjects (
course_id INT NOT NULL, subject_id INT NOT NULL, semester TINYINT NOT NULL,
academic_year VARCHAR(9) NOT NULL, PRIMARY KEY(course_id,subject_id,semester,academic_year),
CHECK(semester BETWEEN 1 AND 12), FOREIGN KEY(course_id) REFERENCES courses(course_id),
FOREIGN KEY(subject_id) REFERENCES subjects(subject_id)) ENGINE=InnoDB""",
"""CREATE TABLE IF NOT EXISTS assessments (
assessment_id INT AUTO_INCREMENT PRIMARY KEY, course_id INT NOT NULL, subject_id INT NOT NULL,
semester TINYINT NOT NULL, academic_year VARCHAR(9) NOT NULL, assessment_name VARCHAR(80) NOT NULL,
max_marks DECIMAL(6,2) NOT NULL, assessment_date DATE,
UNIQUE KEY uq_assessment(course_id,subject_id,semester,academic_year,assessment_name),
UNIQUE KEY uq_assessment_course(assessment_id,course_id),
CHECK(semester BETWEEN 1 AND 12), CHECK(max_marks > 0),
FOREIGN KEY(course_id,subject_id,semester,academic_year)
REFERENCES course_subjects(course_id,subject_id,semester,academic_year)) ENGINE=InnoDB""",
"""CREATE TABLE IF NOT EXISTS marks (
student_id INT NOT NULL, assessment_id INT NOT NULL, course_id INT NOT NULL,
marks_obtained DECIMAL(6,2) NOT NULL, PRIMARY KEY(student_id,assessment_id),
CHECK(marks_obtained >= 0),
FOREIGN KEY(student_id,course_id) REFERENCES students(student_id,course_id)
ON UPDATE CASCADE ON DELETE CASCADE,
FOREIGN KEY(assessment_id,course_id) REFERENCES assessments(assessment_id,course_id)
ON UPDATE CASCADE ON DELETE CASCADE) ENGINE=InnoDB"""
]
conn = get_connection()
cur = conn.cursor()
try:
    for statement in schema:
        cur.execute(statement)
    conn.commit()
    print("All six tables are ready.")
finally:
    cur.close()
    conn.close()

## 6. Seed fictional demo data (safe to rerun)

In [ ]:
def seed_demo_data():
    conn = get_connection()
    cur = conn.cursor()
    try:
        cur.execute(
            "INSERT INTO courses(course_code,course_name) VALUES(%s,%s) "
            "ON DUPLICATE KEY UPDATE course_name=VALUES(course_name)",
            ("BTECH-ECE", "B.Tech Electronics and Communication Engineering"))
        cur.execute("SELECT course_id FROM courses WHERE course_code=%s", ("BTECH-ECE",))
        course_id = cur.fetchone()[0]

        subjects = [
            ("ECE101","Digital Logic Design"), ("ECE102","Electronic Devices"),
            ("ECE103","Programming in C"), ("ECE104","Network Theory")
        ]
        for code_, name in subjects:
            cur.execute("INSERT INTO subjects(subject_code,subject_name) VALUES(%s,%s) "
                        "ON DUPLICATE KEY UPDATE subject_name=VALUES(subject_name)", (code_,name))
        cur.execute("SELECT subject_id,subject_code FROM subjects WHERE subject_code IN (%s,%s,%s,%s)",
                    tuple(s[0] for s in subjects))
        sub_ids = {code_: sid for sid, code_ in cur.fetchall()}
        year, semester = "2026-2027", 3
        for code_, _ in subjects:
            cur.execute("INSERT IGNORE INTO course_subjects VALUES(%s,%s,%s,%s)",
                        (course_id, sub_ids[code_], semester, year))
            cur.execute(
                "INSERT INTO assessments(course_id,subject_id,semester,academic_year,assessment_name,max_marks,assessment_date) "
                "VALUES(%s,%s,%s,%s,%s,%s,%s) ON DUPLICATE KEY UPDATE max_marks=VALUES(max_marks)",
                (course_id,sub_ids[code_],semester,year,"Internal Assessment 1",100,"2026-10-15"))

        students = [
            ("ECE2601","Aarav Sharma","aarav.demo@example.com"),
            ("ECE2602","Diya Patel","diya.demo@example.com"),
            ("ECE2603","Arjun Rao","arjun.demo@example.com"),
            ("ECE2604","Meera Nair","meera.demo@example.com"),
            ("ECE2605","Ishaan Gupta","ishaan.demo@example.com"),
            ("ECE2606","Ananya Iyer","ananya.demo@example.com"),
            ("ECE2607","Kabir Singh","kabir.demo@example.com"),
            ("ECE2608","Sara Khan","sara.demo@example.com"),
            ("ECE2609","Vivaan Das","vivaan.demo@example.com"),
            ("ECE2610","Nisha Reddy","nisha.demo@example.com")
        ]
        for roll,name,email in students:
            cur.execute(
                "INSERT INTO students(roll_number,student_name,email,course_id) VALUES(%s,%s,%s,%s) "
                "ON DUPLICATE KEY UPDATE student_name=VALUES(student_name),email=VALUES(email),course_id=VALUES(course_id)",
                (roll,name,email,course_id))
        cur.execute("SELECT student_id,roll_number FROM students WHERE roll_number LIKE 'ECE26%'")
        student_ids = {roll: sid for sid,roll in cur.fetchall()}
        cur.execute("SELECT assessment_id,subject_id FROM assessments WHERE course_id=%s AND semester=%s AND academic_year=%s",
                    (course_id,semester,year))
        assessment_ids = {sid: aid for aid,sid in cur.fetchall()}
        scores = [
            [82,78,91,75],[91,88,86,90],[74,80,79,77],[88,93,95,89],[69,72,81,70],
            [95,90,92,94],[77,75,73,82],[84,86,89,87],[63,71,68,74],[90,85,88,92]
        ]
        for student, row_scores in zip(students, scores):
            for (subject_code,_), score in zip(subjects,row_scores):
                cur.execute(
                    "INSERT INTO marks(student_id,assessment_id,course_id,marks_obtained) VALUES(%s,%s,%s,%s) "
                    "ON DUPLICATE KEY UPDATE marks_obtained=VALUES(marks_obtained)",
                    (student_ids[student[0]],assessment_ids[sub_ids[subject_code]],course_id,score))
        conn.commit()
        print("Demo records inserted/updated.")
    except Exception:
        conn.rollback()
        raise
    finally:
        cur.close()
        conn.close()

seed_demo_data()

## 7. CRUD: create, read, update, delete

In [ ]:
def add_student(roll_number, student_name, email=None, course_code="BTECH-ECE"):
    if not roll_number.strip() or not student_name.strip():
        raise ValueError("Roll number and name are required.")
    conn = get_connection()
    cur = conn.cursor()
    try:
        cur.execute("SELECT course_id FROM courses WHERE course_code=%s", (course_code,))
        row = cur.fetchone()
        if row is None:
            raise ValueError("Course code not found.")
        cur.execute("INSERT INTO students(roll_number,student_name,email,course_id) VALUES(%s,%s,%s,%s)",
                    (roll_number.strip(),student_name.strip(),email or None,row[0]))
        conn.commit()
        print("Added student ID:",cur.lastrowid)
    except Exception:
        conn.rollback()
        raise
    finally:
        cur.close()
        conn.close()

def list_students():
    conn = get_connection()
    try:
        return pd.read_sql_query(
            "SELECT s.student_id,s.roll_number,s.student_name,s.email,c.course_name "
            "FROM students s JOIN courses c ON s.course_id=c.course_id ORDER BY s.roll_number", conn)
    finally:
        conn.close()

def update_student_name(roll_number, new_name):
    conn = get_connection()
    cur = conn.cursor()
    try:
        cur.execute("UPDATE students SET student_name=%s WHERE roll_number=%s",
                    (new_name.strip(),roll_number.strip()))
        conn.commit()
        print("Rows updated:",cur.rowcount)
    except Exception:
        conn.rollback()
        raise
    finally:
        cur.close()
        conn.close()

def delete_student(roll_number):
    # WARNING: deleting a student also deletes their marks through ON DELETE CASCADE.
    conn = get_connection()
    cur = conn.cursor()
    try:
        cur.execute("DELETE FROM students WHERE roll_number=%s",(roll_number.strip(),))
        conn.commit()
        print("Rows deleted:",cur.rowcount)
    except Exception:
        conn.rollback()
        raise
    finally:
        cur.close()
        conn.close()

display(list_students())

## 8. Report: marks and percentage

In [ ]:
def marks_report(year="2026-2027", semester=3):
    conn = get_connection()
    try:
        return pd.read_sql_query(
            "SELECT s.roll_number,s.student_name,sub.subject_name,a.assessment_name,"
            "m.marks_obtained,a.max_marks,ROUND(100.0*m.marks_obtained/a.max_marks,2) AS percentage "
            "FROM marks m JOIN students s ON s.student_id=m.student_id "
            "JOIN assessments a ON a.assessment_id=m.assessment_id "
            "JOIN subjects sub ON sub.subject_id=a.subject_id "
            "WHERE a.academic_year=%s AND a.semester=%s ORDER BY s.roll_number,sub.subject_name",
            conn, params=(year,semester))
    finally:
        conn.close()

report = marks_report()
display(report.head(12))

## 9. Report: subject-wise average percentage

In [ ]:
def subject_averages(year="2026-2027", semester=3):
    conn = get_connection()
    try:
        return pd.read_sql_query(
            "SELECT sub.subject_name,ROUND(AVG(100.0*m.marks_obtained/a.max_marks),2) AS average_percentage,"
            "COUNT(*) AS recorded_marks FROM marks m "
            "JOIN assessments a ON a.assessment_id=m.assessment_id "
            "JOIN subjects sub ON sub.subject_id=a.subject_id "
            "WHERE a.academic_year=%s AND a.semester=%s "
            "GROUP BY sub.subject_id,sub.subject_name ORDER BY average_percentage DESC",
            conn, params=(year,semester))
    finally:
        conn.close()

subject_avg = subject_averages()
display(subject_avg)

## 10. Report: student ranking

In [ ]:
def student_rankings(year="2026-2027", semester=3):
    conn = get_connection()
    try:
        return pd.read_sql_query(
            "SELECT s.roll_number,s.student_name,"
            "ROUND(AVG(100.0*m.marks_obtained/a.max_marks),2) AS average_percentage,"
            "COUNT(*) AS assessments_count FROM students s "
            "JOIN marks m ON m.student_id=s.student_id "
            "JOIN assessments a ON a.assessment_id=m.assessment_id "
            "WHERE a.academic_year=%s AND a.semester=%s "
            "GROUP BY s.student_id,s.roll_number,s.student_name "
            "ORDER BY average_percentage DESC,s.roll_number",
            conn, params=(year,semester))
    finally:
        conn.close()

rankings = student_rankings()
display(rankings)

## 11. Graph 1 — class average by subject

In [ ]:
plt.figure(figsize=(9,5))
plt.bar(subject_avg["subject_name"],subject_avg["average_percentage"])
plt.title("Class Average by Subject")
plt.xlabel("Subject")
plt.ylabel("Average percentage (%)")
plt.ylim(0,100)
plt.xticks(rotation=20,ha="right")
plt.tight_layout()
plt.show()

## 12. Graph 2 — student ranking

In [ ]:
top = rankings.head(10).sort_values("average_percentage")
plt.figure(figsize=(9,5))
plt.barh(top["student_name"],top["average_percentage"])
plt.title("Student Performance — Average Percentage")
plt.xlabel("Average percentage (%)")
plt.ylabel("Student")
plt.xlim(0,100)
plt.tight_layout()
plt.show()

## 13. Graph 3 — marks distribution

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(report["percentage"],bins=10,edgecolor="black")
plt.title("Distribution of Assessment Percentages")
plt.xlabel("Percentage (%)")
plt.ylabel("Number of recorded marks")
plt.tight_layout()
plt.show()

## 14. Data-quality checks

In [ ]:
conn = get_connection()
cur = conn.cursor()
try:
    cur.execute("SELECT COUNT(*) FROM students")
    print("Students:",cur.fetchone()[0])
    cur.execute("SELECT COUNT(*) FROM subjects")
    print("Subjects:",cur.fetchone()[0])
    cur.execute("SELECT COUNT(*) FROM marks m JOIN assessments a ON a.assessment_id=m.assessment_id "
                "WHERE m.marks_obtained < 0 OR m.marks_obtained > a.max_marks")
    print("Invalid marks (expected 0):",cur.fetchone()[0])
finally:
    cur.close()
    conn.close()

## Short viva explanation

**Aim:** Store student, course, subject, assessment and marks data in a relational database, then generate useful academic reports.

**Technology:** MySQL stores records; Python connects using MySQL Connector/Python; Pandas structures query results; Matplotlib plots graphs; Colab is the development environment.

**Design:** Six tables separate courses, students, subjects, course-subject offerings, assessments and marks. Primary keys identify records; foreign keys maintain relationships. The composite primary key `(student_id, assessment_id)` prevents duplicate mark entries for the same student and assessment. Composite foreign keys also prevent mixing students and assessments from different courses.

**Normalization:** Details are stored once and linked using IDs, reducing repetition and update anomalies. This design aims for 3NF.

**Features:** Student CRUD, mark-sheet report, subject average, student ranking and graphs.

**Security:** Parameterized queries keep values separate from SQL code; constraints and transactions improve data integrity. Credentials are stored in Colab Secrets.

**Limitations/future scope:** Demo data is fictional. Future work can add login roles, attendance, multiple assessments, CSV/PDF exports and a Streamlit/web interface.

**Likely questions**
- **Primary key?** A field or group of fields uniquely identifying a row.
- **Foreign key?** A constraint linking a row to an existing row in another table.
- **Normalization?** Structuring tables to reduce duplication and data anomalies.
- **Why parameterized queries?** Safer handling of values and reduced SQL-injection risk.
- **Why percentage?** `marks_obtained / max_marks × 100` allows comparison when maximum marks differ.
- **Why graphs?** They make class trends and differences between subjects/students easier to understand.